<div class="alert alert-block alert-success">

# App 5: 向量数据库构建与 RAG 架构准备 (Vector Database Build)

**项目:** FIT5196 A1 (Extended Part)  
**模块:** App 5 - 检索增强生成 (RAG) 外部记忆库构建  
**作者:** Zihan Yin  
**日期:** 2026.02.28
**环境:** Google Colab (A00 GPU)

</div>

**概览 (Overview):**  
本 Notebook 的目标是为基于 Llama-3 的语言模型构建外部检索数据库，以支持 RAG (检索增强生成) 架构。通过建立专门的向量检索系统，限制模型在生成回答时只能依赖于检索到的上下文，从而降低产生幻觉 (Hallucination) 的概率。

<div class="alert alert-block alert-info">

## 核心技术选型与架构设计

**1. Embedding 模型选型: `BAAI/bge-large-en-v1.5`**
* 该模型将文本映射为 1024 维的稠密向量。相比于低维模型，高维向量空间能保留更多的语义特征，有助于提升不同表达方式（如同义词、相关概念）之间的检索召回率。

**2. 双轨制检索架构 (Dual-Track Architecture):**
考虑到数据完整性与检索逻辑的区别，本系统采用分离式的双轨存储：
* **语义检索轨 (ChromaDB):** 存放约 212 万条经过预处理的顾客评论。以单条评论为粒度进行向量化，并附加商铺属性作为元数据 (Metadata)，用于检索时的条件过滤。
* **精确查找轨 (Parquet Lookup Table):** 存放约 51.3 万家加州商铺的结构化特征信息（包含无评论记录的商铺）。作为键值查找表，在推理阶段提供确切的 URL、地址等客观信息，作为 RAG 系统的基础数据兜底。

</div>

## Step 1: 环境配置与云盘挂载 (Environment Setup)

**目标:** 搭建支持向量计算与本地数据库持久化的运行环境。

**技术细节:**
1. 安装 `chromadb` 作为本地向量数据库后端，安装 `sentence-transformers` 用于加载 Embedding 模型。
2. 规避库版本冲突：移除安装命令中对 `pandas` 的强制升级，保留 Colab 原生兼容版本，以确保 Google Drive 能够正常挂载。

In [ ]:
# ==========================================
# [Cell 1] 环境配置与 Google Drive 挂载
# ==========================================
# 移除 pandas 升级，保留 Colab 原生兼容版本，同时限制部分底层库以防冲突
!pip install -q -U chromadb sentence-transformers pyarrow fastparquet

import os
import pandas as pd
import torch
import chromadb
from sentence_transformers import SentenceTransformer
from google.colab import drive

# 挂载 Google Drive
drive.mount('/content/drive')

# --- 路径配置 ---
BASE_DIR = '/content/drive/MyDrive/FIT5196_A1_Extension/App5'

# 输入文件路径
INPUT_REVIEWS_PATH = os.path.join(BASE_DIR, 'Task1_for_GroupALL.ftr')
INPUT_METADATA_PATH = os.path.join(BASE_DIR, 'meta-California.json')

# 输出文件与数据库路径
CHROMADB_DIR = os.path.join(BASE_DIR, 'ChromaDB')
METADATA_LOOKUP_PATH = os.path.join(BASE_DIR, 'gmap_metadata_lookup.parquet')

print("环境依赖安装完毕，路径映射初始化成功。")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/

## Step 2: 数据读取、清洗与精确查找轨构建 (Data Loading & Lookup Table Build)

**目标:** 提取有效评论数据，并构建用于结构化查找的 Parquet 静态表。

**执行逻辑:**
1. **复用预处理:** 读取全量评论数据，通过阈值 (`word_count >= 10`) 筛选出约 212 万条有效长评论。
2. **商铺去重:** 对 `meta-California.json` 中的数据按 `gmap_id` 执行去重 (`drop_duplicates`)，确保索引的唯一性。
3. **序列化落盘:** 将全量商铺的静态字段 (`gmap_id`, `name`, `category`, `description`, `address`, `avg_rating`, `url`) 提取并保存为 `gmap_metadata_lookup.parquet` 文件，用于后续的 `O(1)` 精确信息查找。

In [ ]:
# ==========================================
# [Cell 2] 数据加载与清洗 (复用预处理逻辑)
# ==========================================
print("正在加载并清洗评论数据 (Reviews)...")
gmap_reviews = pd.read_feather(INPUT_REVIEWS_PATH)
initial_len = len(gmap_reviews)

gmap_reviews = gmap_reviews.dropna(subset=['review_text'])
# 动态计算单词数并过滤短文本
gmap_reviews['word_count'] = gmap_reviews['review_text'].apply(lambda x: len(str(x).split()))
gmap_reviews = gmap_reviews[gmap_reviews['word_count'] >= 10]
gmap_reviews = gmap_reviews[['gmap_id', 'user_id', 'review_text', 'word_count']]

valid_gmap_ids_for_sft = set(gmap_reviews['gmap_id'].unique())
print(f"评论数据 gmap_reviews 清洗完毕。原始: {initial_len} -> 有效长文本: {len(gmap_reviews)}")
print(f"   拥有有效长评论的商铺数量: {len(valid_gmap_ids_for_sft)}\n")


print("正在加载并清洗商铺元数据 (Metadata)...")
gmap_metadata = pd.read_json(INPUT_METADATA_PATH, lines=True)
initial_meta_len = len(gmap_metadata)

cols_to_keep = ['gmap_id', 'name', 'category', 'description', 'address', 'avg_rating', 'url']
gmap_metadata = gmap_metadata[cols_to_keep]
gmap_metadata = gmap_metadata.dropna(subset=['name'])

# 源头去重，确保索引绝对唯一
gmap_metadata = gmap_metadata.drop_duplicates(subset=['gmap_id'], keep='last')
gmap_metadata = gmap_metadata.set_index('gmap_id')

print(f"元数据 gmap_metadata 加载完毕。原始: {initial_meta_len} -> 有效全局商铺数: {len(gmap_metadata)}")

# --- 架构落地 1：保存全量元数据作为精确查找轨 ---
gmap_metadata.to_parquet(METADATA_LOOKUP_PATH)
print(f"全量商铺查找表已保存至: {METADATA_LOOKUP_PATH} (后续 RAG 推理兜底使用)")

正在加载并清洗评论数据 (Reviews)...
评论数据 gmap_reviews 清洗完毕。原始: 5767463 -> 有效长文本: 2125904
   拥有有效长评论的商铺数量: 29088

正在加载并清洗商铺元数据 (Metadata)...
元数据 gmap_metadata 加载完毕。原始: 515961 -> 有效全局商铺数: 513124
全量商铺查找表已保存至: /content/drive/MyDrive/FIT5196_A1_Extension/App5/gmap_metadata_lookup.parquet (后续 RAG 推理兜底使用)


## Step 3: 数据特征融合 (Data Fusion & Metadata Formatting)

**目标:** 将商铺的结构化属性合并至评论数据中，构建 ChromaDB 所需的数据格式。

**执行逻辑:**
为实现精确的范围检索，需要将商铺的 `gmap_id`、`name` 和 `category` 拼接到对应的每一条评论记录中。在后续的检索阶段，系统可以通过这些元数据进行前置过滤 (Metadata Filtering)，确保只在特定商铺的评论子集中计算向量相似度，防止检索结果出现跨店混淆。

In [ ]:
# ==========================================
# [Cell 3] 融合商铺信息至评论，构建向量入库 Payload
# ==========================================
print("正在将商铺元数据关联至对应的评论记录中...")

# 为了构建 ChromaDB 的 Metadata，我们需要把商铺的 name, category 等信息 join 到 review 表中
# reset_index 以便 gmap_id 参与 join
metadata_flat = gmap_metadata.reset_index()[['gmap_id', 'name', 'category', 'avg_rating']]

# 类别信息通常是一个列表，转化为字符串以符合 ChromaDB Metadata 格式化要求
metadata_flat['category'] = metadata_flat['category'].apply(lambda x: ", ".join(x) if isinstance(x, list) else str(x))

# 将商铺信息合并到评论中 (使用 inner join，确保入库的评论都有对应的商铺信息)
gmap_reviews_with_meta = pd.merge(gmap_reviews, metadata_flat, on='gmap_id', how='inner')

# 为每一条数据生成全局唯一的 ID (Document ID) 供 ChromaDB 使用
gmap_reviews_with_meta['doc_id'] = gmap_reviews_with_meta.index.astype(str) + "_" + gmap_reviews_with_meta['gmap_id']

print(f"融合完成。准备入库的独立向量文档数量: {len(gmap_reviews_with_meta)}")
display(gmap_reviews_with_meta.head(2))

正在将商铺元数据关联至对应的评论记录中...
融合完成。准备入库的独立向量文档数量: 2125904


,gmap_id,user_id,review_text,word_count,name,category,avg_rating,doc_id
0,0x54d46d1125349d73:0x2ab5724cf1cbc511,112420076785194463877,We met with friends at this restaurant for din...,54,Mingos Sports Bar,Cocktail bar,4.2,0_0x54d46d1125349d73:0x2ab5724cf1cbc511
1,0x54d46d1125349d73:0x2ab5724cf1cbc511,103079253096551262298,Best place on earth !!! Good staff cool little...,14,Mingos Sports Bar,Cocktail bar,4.2,1_0x54d46d1125349d73:0x2ab5724cf1cbc511


## Step 4: 批量向量化与 ChromaDB 入库 (Batch Embedding & Upserting)

**目标:** 使用 GPU 批量计算文本的 1024 维语义向量，并将数据写入本地 ChromaDB 实例。

**执行逻辑:**
1. **规避 I/O 瓶颈:** 为避免直接读写 Google Drive 带来的网络 I/O 延迟，先在 Colab 本地磁盘路径构建数据库，完成后再整体打包迁移至云盘。
2. **控制批次大小:** ChromaDB 的 SQLite 底层对单次插入的 SQL 变量数量有硬性限制。将 `BATCH_SIZE` 设定为 5000，以保证写入过程的稳定性。
3. **更新或插入 (Upsert):** 采用 `upsert` 操作。若程序中断后重启，已存在的 `doc_id` 记录会被覆盖更新，避免产生重复数据。

In [ ]:
# ==========================================
# [Cell 4] 加载模型、本地极速入库并打包至云盘
# ==========================================
from tqdm.auto import tqdm
import shutil
import time

# 1. 启动硬件加速与加载模型
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"正在使用计算引擎: {device.upper()}")

EMBEDDING_MODEL_ID = "BAAI/bge-large-en-v1.5"
print(f"加载 Embedding 模型: {EMBEDDING_MODEL_ID} ...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_ID, device=device)

# --- 【终极优化】使用 Colab 本地磁盘代替 Google Drive 进行高频读写 ---
LOCAL_CHROMADB_DIR = '/content/ChromaDB_local'
# -------------------------------------------------------------------

# 2. 初始化本地持久化 ChromaDB
client = chromadb.PersistentClient(path=LOCAL_CHROMADB_DIR)
collection_name = "gmap_reviews_collection"

# 获取或创建 Collection
collection = client.get_or_create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}
)
print(f"ChromaDB Collection '{collection_name}' 准备就绪。")

# 3. 分批次高性能入库逻辑
BATCH_SIZE = 5000
total_docs = len(gmap_reviews_with_meta)

print(f"\n🚀 开始执行向量化与极速本地入库任务 (总计 {total_docs} 条)...")
start_time = time.time()

for i in tqdm(range(0, total_docs, BATCH_SIZE), desc="Upserting to Local ChromaDB"):
    batch_df = gmap_reviews_with_meta.iloc[i : i + BATCH_SIZE]

    batch_ids = batch_df['doc_id'].tolist()
    batch_texts = batch_df['review_text'].tolist()

    batch_metadata = []
    for _, row in batch_df.iterrows():
        meta = {
            "gmap_id": str(row['gmap_id']),
            "store_name": str(row['name']),
            "category": str(row['category']),
            "avg_rating": float(row['avg_rating']) if pd.notnull(row['avg_rating']) else 0.0,
            "user_id": str(row['user_id'])
        }
        batch_metadata.append(meta)

    # 增加模型处理 batch_size
    batch_embeddings = embedding_model.encode(batch_texts, batch_size=512, show_progress_bar=False, normalize_embeddings=True)

    collection.upsert(
        ids=batch_ids,
        documents=batch_texts,
        embeddings=batch_embeddings.tolist(),
        metadatas=batch_metadata
    )

end_time = time.time()
print(f"\n极速入库完成！耗时: {(end_time - start_time)/60:.2f} 分钟。")
print(f"目前数据库的总记录数为: {collection.count()}")

# 4. 最终将本地构建好的数据库整体迁移至 Google Drive
print("\n正在将构建好的数据库整体迁移至 Google Drive (这可能需要几分钟)...")
# 我们之前 Cell 1 定义的 CHROMADB_DIR 就是 Google Drive 路径
if os.path.exists(CHROMADB_DIR):
    print("检测到云盘中已有旧数据库文件夹，正在清理...")
    shutil.rmtree(CHROMADB_DIR)

shutil.copytree(LOCAL_CHROMADB_DIR, CHROMADB_DIR)
print(f"迁移成功！向量数据库已永久保存在您的云盘: {CHROMADB_DIR}")

正在使用计算引擎: CUDA
加载 Embedding 模型: BAAI/bge-large-en-v1.5 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

ChromaDB Collection 'gmap_reviews_collection' 准备就绪。

🚀 开始执行向量化与极速本地入库任务 (总计 2125904 条)...


Upserting to Local ChromaDB:   0%|          | 0/426 [00:00<?, ?it/s]


极速入库完成！耗时: 190.16 分钟。
目前数据库的总记录数为: 2125904

正在将构建好的数据库整体迁移至 Google Drive (这可能需要几分钟)...
迁移成功！向量数据库已永久保存在您的云盘: /content/drive/MyDrive/FIT5196_A1_Extension/App5/ChromaDB


## Step 5: 向量检索基础验证 (Basic Sanity Check)

**目标:**   
验证 ChromaDB 集合的写入完整性及元数据过滤 (Metadata Filtering) 功能。

**测试方案:**  
设定目标商铺的 `gmap_id` 作为过滤条件，输入特定查询语句。验证系统能否将文本正确转化为向量，并在限定的商铺评论池中，基于余弦相似度召回相关的 Top-K 评论记录。

**测试案例与过关标准:**
* **案例来源:** 沿用 `03_LLM_Validation.ipynb` (Step 3) 中的测试集 Sample 1 (商铺名称: Great Clips)。
* **过关标准:** 将目标商铺的 `gmap_id` 注入过滤条件，执行查询 "What are the experiences with specific stylists like Mona?"。若 ChromaDB 返回的 Top-K 结果中，精准包含了原始数据集中实际提及理发师 "Mona" 的评论文本，则证明商铺级的元数据隔离 (Metadata Filtering) 与余弦相似度匹配逻辑运行正常。

In [ ]:
# ==========================================
# [Cell 5] 向量库检索质量独立验证 (Sanity Check)
# ==========================================

print("="*60)
print("开始执行语义检索测试...")

# 模拟用户的复杂提问
test_query = "What are the experiences with specific stylists like Mona?"
# 假定我们从前端获得了用户想查询的具体商铺 ID (来自之前验证集的 Great Clips 样本)
target_gmap_id = "0x80dd4b81ab9e529f:0x9e02fbe40e706d19"

# 1. 对用户 Query 提取向量
query_embedding = embedding_model.encode([test_query], normalize_embeddings=True).tolist()

# 2. 在 ChromaDB 中执行带条件过滤 (Metadata Filtering) 的向量检索
results = collection.query(
    query_embeddings=query_embedding,
    n_results=3, # 取 Top-3 最相关的评论
    where={"gmap_id": target_gmap_id} # 极其关键：通过元数据过滤，只在该店铺内搜索
)

print(f"针对问题: '{test_query}'")
print(f"检索范围限制: 商铺 ID [{target_gmap_id}]\n")

for idx, (doc, meta, dist) in enumerate(zip(results['documents'][0], results['metadatas'][0], results['distances'][0])):
    print(f"--- Top {idx+1} 匹配结果 (距离/相似度倒数: {dist:.4f}) ---")
    print(f"商铺名称: {meta['store_name']}")
    print(f"评论内容: {doc}\n")

print("="*60)

开始执行语义检索测试...
针对问题: 'What are the experiences with specific stylists like Mona?'
检索范围限制: 商铺 ID [0x80dd4b81ab9e529f:0x9e02fbe40e706d19]

--- Top 1 匹配结果 (距离/相似度倒数: 0.2516) ---
商铺名称: Great Clips
评论内容: Excellent atmosphere. One of the stylist named Luis transformed three ladies styles from plain to fabulous in the short time was waiting. So I then decided to get my hair done as well❣️Today I walked in and my hairstylist was unavailable so I tried a new lady named Mona. She was super fun and gave me an excellent hairstyle with instructions for the aftercare ( which in the past, turned out to be a nightmare) and an easy timeless style. You rock Mona

--- Top 2 匹配结果 (距离/相似度倒数: 0.2868) ---
商铺名称: Great Clips
评论内容: Hi I went in the salon and my regular person was not in, so I had a lady by the name of Mona, do my hair.  Somehow my information in the computer was missing so it didn't have the cut I usually get. Mona said she knew what I was talking about and she started cutting, she was so nice a

### 验证结果评价

**评估结论: 测试通过**

通过分析检索输出，当前 ChromaDB 实例的运行状况符合预期设计：
1. **元数据隔离成功 (Metadata Filtering):** 所有返回的 Top-3 结果其商铺名称均被严格限定为 "Great Clips"，证明 `where={"gmap_id": target_gmap_id}` 过滤条件生效，杜绝了跨店铺的上下文污染。
2. **余弦相似度匹配精准:** 针对提问中包含的实体词 "Mona"，系统准确地将原数据集中真实提及 "Mona" 的两条评论排在了 Top 1 (距离 0.2516) 和 Top 2 (距离 0.2868)。
3. **容错与排序逻辑合理:** Top 3 返回了另一位理发师 (Lourdes) 的评价 (距离 0.3260)。这符合向量检索的逻辑：在穷尽了所有包含目标实体 ("Mona") 的文本后，系统退而求其次，召回了语义上最接近“具体理发师体验”的评论。

**总结:** 系统的基础向量入库与带有元数据过滤的语义检索流水线已成功跑通。

## Step 6: 双轨制架构联合验证 (Dual-Track Architecture Verification)

**目标:**   
模拟 RAG 系统的实际数据流，验证双轨检索机制的协同工作状态。

**测试逻辑:**
1. **精确查找轨 (Track 1):** 测试 Pandas 能否通过指定的 `gmap_id` 正确提取该商铺的地址与链接信息。
2. **语义理解测试 (Track 2):** 在提问中避免使用评论原文的确切关键词（如用 "pets" 代替 "dog"），验证 Embedding 模型对同义概念和相关语义的跨词汇召回能力。

**测试案例与过关标准:**
* **案例来源:** 沿用 `03_LLM_Validation.ipynb` (Step 3) 中的测试集 Sample 2 (商铺名称: Vape Prodigy)。
* **过关标准:**
  1. **静态查找过关:** 能够依据给定的 `gmap_id`，从 Parquet 查找表中瞬间返回 Vape Prodigy 的确切地址与 URL 等基础信息。
  2. **语义检索过关:** 使用抽象重写后的提问 ("...entertainment facilities..." 以及 "...policy regarding pets...") 进行检索。若系统返回的 Top-K 结果成功召回了包含原始词汇 "arcade machines" (街机) 和 "dog-friendly" (狗友好) 的评论，则证明高维 Embedding 模型成功建立了同义概念的隐式语义映射，向量数据库的构建与预期目标完全相符。

In [3]:
# ==========================================
# [Cell 6] 进阶验证：双轨制架构联合测试
# ==========================================
import pandas as pd

print("="*60)
print("开始执行双轨制架构联合测试 (Dual-Track Test)...")

# --- 1. 测试精确查找轨 (Metadata Lookup) ---
print("\n[Track 1] 验证静态元数据查找表 (Pandas Lookup)...")
# 读取我们在 Cell 2 中保存的 parquet 文件
lookup_df = pd.read_parquet(METADATA_LOOKUP_PATH)

# 使用 App 5 验证集中的 Sample 2: Vape Prodigy 的 gmap_id
test_gmap_id = "0x80c2cd7de86db263:0x8dc0e3873986273b"

if test_gmap_id in lookup_df.index:
    store_info = lookup_df.loc[test_gmap_id]
    print(f"瞬间命中精确查找表！(O(1) 复杂度)")
    print(f"  - 商铺名称: {store_info['name']}")
    print(f"  - 商铺分类: {store_info['category']}")
    print(f"  - 商铺地址: {store_info['address']}")
    print(f"  - 商铺链接: {store_info['url']}")
else:
    print("查找失败，请检查 Parquet 文件。")

# --- 2. 测试语义检索轨 (ChromaDB 隐式语义理解) ---
print("\n[Track 2] 验证 ChromaDB 与 BAAI 模型的隐式语义理解 (Implicit Semantics)...")

# 故意不使用原文中的 "dog" 或 "arcade machines"，而是使用更宽泛的询问词
complex_query = "Are there any entertainment facilities inside the shop, and what is their policy regarding pets or animals?"

# 将问题向量化
query_embedding = embedding_model.encode([complex_query], normalize_embeddings=True).tolist()

# 执行向量检索
results = collection.query(
    query_embeddings=query_embedding,
    n_results=2, # 取 Top-2 看看能否同时命中包含狗和街机的评论
    where={"gmap_id": test_gmap_id}
)

print(f"\n用户提问 (User Query): '{complex_query}'")
for idx, (doc, dist) in enumerate(zip(results['documents'][0], results['distances'][0])):
    print(f"--- Top {idx+1} 匹配结果 (距离/相似度倒数: {dist:.4f}) ---")
    print(f"评论原文: {doc}\n")

print("="*60)
print("验证通过标准：如果上述输出了正确的商铺地址/URL，并且成功找出了包含 'dog-friendly' 或 'arcade' 的评论，说明 RAG 检索底层已完美搭建！")

开始执行双轨制架构联合测试 (Dual-Track Test)...

[Track 1] 验证静态元数据查找表 (Pandas Lookup)...
瞬间命中精确查找表！(O(1) 复杂度)
  - 商铺名称: Vape Prodigy
  - 商铺分类: ['Cigar shop' 'Hookah bar']
  - 商铺地址: Vape Prodigy, 9510 Firestone Blvd, Downey, CA 90241
  - 商铺链接: https://www.google.com/maps/place//data=!4m2!3m1!1s0x80c2cd7de86db263:0x8dc0e3873986273b?authuser=-1&hl=en&gl=us

[Track 2] 验证 ChromaDB 与 BAAI 模型的隐式语义理解 (Implicit Semantics)...

用户提问 (User Query): 'Are there any entertainment facilities inside the shop, and what is their policy regarding pets or animals?'
--- Top 1 匹配结果 (距离/相似度倒数: 0.3601) ---
评论原文: The gentleman in this store were very helpful and accommodating. They really knew their stuff, and totally helped me out when I needed advice and direction in what to get. I highly recommend the store , there's a great selection! And to top it off, they are dog friendly!

--- Top 2 匹配结果 (距离/相似度倒数: 0.3639) ---
评论原文: I'm "SUPER" picky and a hard to please guy and this is the only vape shop I found that I really like. 

### 验证结果评价

**评估结论: 测试通过 (Pass)**

本环节的输出数据充分证明了“双轨制架构”在实际复杂场景下的优越性：
1. **Track 1 (精确查找表) 验证:** Pandas 依据给定的 `gmap_id`，在 `O(1)` 时间内成功读取并输出了 Vape Prodigy 的类别、地址与对应 Google Maps URL。证明了静态特征兜底机制的可用性。
2. **Track 2 (隐式语义理解) 验证:** 面对重写后高度抽象的用户提问 ("entertainment facilities" 与 "policy regarding pets")，系统成功绕过了字面关键词匹配的局限。
   * Top 1 结果成功召回了包含 "dog friendly" 的评论 (准确映射了 pets/animals)。
   * Top 2 结果成功召回了包含 "arcade machines" 的评论 (准确映射了 entertainment facilities)。

**总结:** 1024 维的 `bge-large-en-v1.5` Embedding 模型展示了卓越的跨词汇语义对齐能力。双轨制 RAG 底层数据库构建工作圆满完成。

---
# END